## RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/Users/harsh/Desktop/Prompt Engg/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all pdfs inside the directory

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir=Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.\n")

    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name}\n")

        except Exception as e:
            print(f"Error processing {pdf_file}: {e}\n")

    print(f"Total pages loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents=process_all_pdfs("../data")

Found 2 PDF files to process.

Processing: IJRAR1ARP035.pdf
Loaded 12 pages from IJRAR1ARP035.pdf

Processing: Dr.R.Praba-StudyonMLAlgorithms.pdf
Loaded 6 pages from Dr.R.Praba-StudyonMLAlgorithms.pdf

Total pages loaded: 18


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2019-09-27T13:46:56+05:30', 'author': 'Students', 'moddate': '2019-09-27T13:46:56+05:30', 'source': '../data/pdf/IJRAR1ARP035.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'IJRAR1ARP035.pdf', 'file_type': 'pdf'}, page_content="© 2019 IJRAR June 2019, Volume 6, Issue 2                                           www.ijrar.org  (E-ISSN 2348-1269, P- ISSN 2349-5138) \nIJRAR1ARP035 International Journal of Research and Analytical Reviews (IJRAR) www.ijrar.org 197 \n \nMACHINE LEARNING \n  S. Geetha Gowri1,  R.Devi 2, Dr.K.Sethuraman M.A., M.Phil., Ph.D.3 \n1,2 Department of Computer Science, Parvathy's Arts and Science College, Dindigul, Tamilnadu. \n3Sky (Simplified Kundalini Yoga), World Community Service Centre, Chennai 3 \nAbstract: \nMachine Learning is the art (and science) of enabling machin es to learn things which are not explicitly programmed. It invol

In [4]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ", ""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...") 
        print(f"\nMetadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 18 documents into 78 chunks

Example chunk:
Content: © 2019 IJRAR June 2019, Volume 6, Issue 2                                           www.ijrar.org  (E-ISSN 2348-1269, P- ISSN 2349-5138) 
IJRAR1ARP035 International Journal of Research and Analytical ...

Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2019-09-27T13:46:56+05:30', 'author': 'Students', 'moddate': '2019-09-27T13:46:56+05:30', 'source': '../data/pdf/IJRAR1ARP035.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'IJRAR1ARP035.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2019-09-27T13:46:56+05:30', 'author': 'Students', 'moddate': '2019-09-27T13:46:56+05:30', 'source': '../data/pdf/IJRAR1ARP035.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'IJRAR1ARP035.pdf', 'file_type': 'pdf'}, page_content="© 2019 IJRAR June 2019, Volume 6, Issue 2                                           www.ijrar.org  (E-ISSN 2348-1269, P- ISSN 2349-5138) \nIJRAR1ARP035 International Journal of Research and Analytical Reviews (IJRAR) www.ijrar.org 197 \n \nMACHINE LEARNING \n  S. Geetha Gowri1,  R.Devi 2, Dr.K.Sethuraman M.A., M.Phil., Ph.D.3 \n1,2 Department of Computer Science, Parvathy's Arts and Science College, Dindigul, Tamilnadu. \n3Sky (Simplified Kundalini Yoga), World Community Service Centre, Chennai 3 \nAbstract: \nMachine Learning is the art (and science) of enabling machin es to learn things which are not explicitly programmed. It invol

### Embeddings and VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str])->np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
## Initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8344.70it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [8]:
class VectorStore:
    def __init__(self, collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to vector store...")

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized with collection: pdf_documents
Existing documents in collection: 0


In [9]:
### Convert the text to embeddings

texts=[doc.page_content for doc in chunks]

embeddings=embedding_manager.generate_embeddings(texts)

# Store in Vector DB

vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 78 texts...


Batches: 100%|██████████| 3/3 [00:00<00:00,  4.49it/s]

Generated embeddings with shape: (78, 384)
Adding 78 documents to vector store...
Successfully added 78 documents to vector store.
Total documents in collection after addition: 78


## Retriever Pipeline from Vector Store

In [ ]:
class RAGRetriever:
    def __init__(self,vector_store: VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
    
    def retrieve(self, query: str, top_k: int=5, score_threshold:float=0.0)->List[Dict[str,Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K:{top_k}, Score Threshold:{score_threshold}\n")
        query_embedding=self.embedding_manager.generate_embeddings([query])[0]

        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i, (doc_id,document, metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    similarity_score=1-distance
                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)\n")
            else:
                print("No documents found")
            return retrieved_docs

        except Exception as e:
            print(f"Error querying vector store: {e}")
            return []
        
rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [11]:
rag_retriever

In [17]:
rag_retriever.retrieve("What is unsupervised Learning?")

Retrieving documents for query: 'What is unsupervised Learning?'
Top K:5, Score Threshold:0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.75it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_62a4e1e1_65',
  'content': 'responding to feedbac k. The field of density \nestimation in statistics, such as calculating the \nprobability density function, is a key application of \nunsupervised learning. Unsupervised learning, on the \nother hand, comprises various domains that need \nsummarising and explaining data aspects. \n \nCluster analysis divides a set of observations into \nsubsets (called clusters) so that observations within \nthe same cluster are comparable based on one or \nmore predetermined criteria, while observations from \ndifferent clusters are distinct. Di fferent clustering \napproaches make different assumptions about the \nstructure of the data, which is commonly \ncharacterized by some similarity metric and \nevaluated, for example, by internal compactness, or \nthe similarity between cluster members, and \nseparation, or the difference between clusters. \nEstimated density and graph connectedness are used \nin other approaches. \n \n \nC. Semi-s

## Integration VectorDB Context Pipeline with LLM Output

In [27]:
### Simple RAG pipeline with Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

## Initialize Groq LLM
groq_api_key=os.getenv("GROQ_API_KEY")
llm=ChatGroq(api_key=groq_api_key, model_name="llama-3.3-70b-versatile", temperature=0.1,max_tokens=1024)

## Simple RAG function: retrive context and generate response
def rag_simple(query, retriever, llm, top_k=3):
    results=retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        print("No relevant context found to answer the query.")
    
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}
        
        Question: {query}
        
        Answer:"""

    response=llm.invoke(prompt.format(context=context, query=query))
    return response.content

In [28]:
answer=rag_simple("What is unsupervised Learning?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is unsupervised Learning?'
Top K:3, Score Threshold:0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.47it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Unsupervised learning is a method of machine learning where the system finds patterns or structure in unlabeled, unclassified, and uncategorized data, without prior knowledge of the output, and learns from it to discover commonalities and relationships in the data.


## Enhanced RAG Pipeline features

In [29]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    results=retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': "No relevant context found.", 'sources': [], 'confidence': 0.0, 'context': ''}
    
    context="\n\n".join([doc['content'] for doc in results])
    
    sources=[{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page':doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview':doc['content'][:300]+'...'
    } for doc in results]
    confidence=max([doc['similarity_score'] for doc in results])

    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}
        
        Question: {query}
        
        Answer:"""
    
    response=llm.invoke(prompt.format(context=context, query=query))

    output={
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context']=context
    return output

results=rag_advanced("What is unsupervised Learning?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", results['answer'],"\n")
print("Sources:", results['sources'],"\n")
print("Confidence:", results['confidence'],"\n")
print("Context Preview:", results['context'][:300])

Retrieving documents for query: 'What is unsupervised Learning?'
Top K:3, Score Threshold:0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.23it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Unsupervised learning is a method of machine learning where the system finds patterns or structure in unlabeled, unclassified, and uncategorized data, without prior knowledge of the output, and learns from it to discover commonalities and react accordingly. 

Sources: [{'source': 'Dr.R.Praba-StudyonMLAlgorithms.pdf', 'page': 2, 'score': 0.29224371910095215, 'preview': 'responding to feedbac k. The field of density \nestimation in statistics, such as calculating the \nprobability density function, is a key application of \nunsupervised learning. Unsupervised learning, on the \nother hand, comprises various domains that need \nsummarising and explaining data aspects. \n \n...'}, {'source': 'IJRAR1ARP035.pdf', 'page': 3, 'score': 0.2406466007232666, 'preview': 'A learning algorithm will receive a set of input instructions along with the corresponding accurate ou tcomes. The learning \nalgorithm will then compare the actual outcome with the accurate outcome and flag an error, if th

In [30]:
## Advanced RAG pipeline: Streaming, citations, history, summarization...

from typing import List,Dict,Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever=retriever
        self.llm=llm
        self.history=[]

    def query(self, question:str, top_k:int=5, min_score:float=0.2, stream:bool=False, summarize:bool=False)->Dict[str,Any]:
        results=self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer="No relevant context found."
            sources=[]
            context=""
        else:
            context="\n\n".join([doc['content'] for doc in results])
            sources=[{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page':doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview':doc['content'][:300]+'...'
            } for doc in results]

            prompt=f"""Use the following context to answer the question concisely.
                Context:
                {context}
                
                Question: {question}
                
                Answer:"""
            
            if stream:
                print("Generating answer (streaming):")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            
            response=self.llm.invoke(prompt.format(context=context, question=question))
            answer=response.content
         
        citations=[f"[{i+1}]{src['source']} (Page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations=answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        summary=None
        if summarize and answer:
            summary_prompt=f"Summarize the following answer in 2 sentences:{answer}"
            summary_resp=self.llm.invoke([summary_prompt])
            summary=summary_resp.content
        
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return{
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

adv_rag=AdvancedRAGPipeline(rag_retriever, llm)
result=adv_rag.query("What is unsupervised Learning?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("\nSummary:", result['summary'])
print("\nHistory:", result['history'][-1])

Retrieving documents for query: 'What is unsupervised Learning?'
Top K:3, Score Threshold:0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.67it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Generating answer (streaming):
Use the following context to answer the question concisely.
                Context:
                responding to feedbac k. The field of density 
estimation in statistics, such as calculating the 
probability density function, is a key ap

plication of 
unsupervised learning. Unsupervised learning, on the 
other hand, comprises various domains that need 
summarising and explaining data aspects. 
 
Cluster analysis divides a set of observations into 
subsets (called clusters) so that observations within 
the same cluster are comparable based on one or 
more predetermined criteria, while observations from 
different clusters are distinct. Di fferent clustering 
approaches make different assumptions about the 
structure of the data, which is commonly 
characterized by some similarity metric and 
evaluated, for example, by internal compactness, or 
the similarity between cluster members, and 
separation, or the difference between clusters. 
Estimated density and graph connectedness are used 
in other approaches. 
 
 
C. Semi-supervised Learning

A learning algorithm will receive a set of input instructions along with the corresponding accurate ou tcomes. The learning 
algorithm will then compare the actual outcome with the a